<a href="https://colab.research.google.com/github/Rajeraghav/AI-Engineer-Journey/blob/main/AI_Chatbot_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [53]:
# ============================================================
#              AI CHATBOT ASSISTANT
#       TENSORFLOW + NLP + BIDIRECTIONAL LSTM
#                    + TF-IDF
# ============================================================

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import re
import random
import pickle
import os

import tensorflow as tf

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Input,
    Embedding,
    Bidirectional,
    LSTM,
    Dense,
    Dropout,
    SpatialDropout1D
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

from tensorflow.keras.optimizers import Adam


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass


# ============================================================
# 3. TITLE
# ============================================================

print("=" * 80)
print("                     AI CHATBOT ASSISTANT")
print("          TENSORFLOW + NLP + BIDIRECTIONAL LSTM")
print("=" * 80)

print("\nTensorFlow version:", tf.__version__)


# ============================================================
# 4. UPLOAD CSV
# ============================================================

print("\nUpload your CSV file.")

print("\nRequired columns:")
print("intent, text, response")

print()

uploaded = files.upload()

if not uploaded:
    raise ValueError("No CSV file was uploaded.")

filename = list(uploaded.keys())[0]

print("\nUploaded file:", filename)


# ============================================================
# 5. LOAD CSV
# ============================================================

df = pd.read_csv(filename)

print("\nCSV loaded successfully!")

print("Original dataset shape:", df.shape)

print("\nOriginal columns:")
print(df.columns.tolist())


# ============================================================
# 6. CLEAN COLUMN NAMES
# ============================================================

df.columns = (
    df.columns
    .astype(str)
    .str.strip()
    .str.lower()
)


# ============================================================
# 7. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = [
    "intent",
    "text",
    "response"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:

    raise ValueError(
        "\nMissing columns: "
        + str(missing_columns)
        + "\n\nYour CSV must contain:\n"
        + "intent,text,response"
    )


# ============================================================
# 8. KEEP REQUIRED COLUMNS
# ============================================================

df = df[
    [
        "intent",
        "text",
        "response"
    ]
].copy()


# ============================================================
# 9. REMOVE NULL VALUES
# ============================================================

print("\nMissing values:")

print(
    df[
        [
            "intent",
            "text",
            "response"
        ]
    ].isnull().sum()
)

df = df.dropna(
    subset=[
        "intent",
        "text",
        "response"
    ]
)


# ============================================================
# 10. CONVERT TO STRING
# ============================================================

for column in [
    "intent",
    "text",
    "response"
]:

    df[column] = (
        df[column]
        .astype(str)
        .str.strip()
    )


# ============================================================
# 11. REMOVE EMPTY VALUES
# ============================================================

df = df[
    (df["intent"] != "") &
    (df["text"] != "") &
    (df["response"] != "")
].copy()


# ============================================================
# 12. REMOVE DUPLICATES
# ============================================================

df = (
    df
    .drop_duplicates(
        subset=[
            "intent",
            "text"
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 13. TEXT PREPROCESSING
# ============================================================

def preprocess_text(text):

    text = str(text).lower()

    # Replace URLs
    text = re.sub(
        r"http\S+|www\S+|https\S+",
        " ",
        text
    )

    # Keep alphabets and numbers
    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


df["clean_text"] = df["text"].apply(
    preprocess_text
)


# ============================================================
# 14. REMOVE EMPTY CLEAN TEXT
# ============================================================

df = df[
    df["clean_text"].str.len() > 0
].reset_index(drop=True)


# ============================================================
# 15. DATASET INFORMATION
# ============================================================

print("\n" + "=" * 80)
print("DATASET INFORMATION")
print("=" * 80)

print(
    "\nDataset shape:",
    df.shape
)

number_of_intents = df["intent"].nunique()

print(
    "\nNumber of intents:",
    number_of_intents
)

print("\nIntent distribution:")

print(
    df["intent"].value_counts()
)


# ============================================================
# 16. DATASET VALIDATION
# ============================================================

if number_of_intents < 2:

    raise ValueError(
        "At least two different intents are required."
    )

minimum_examples = (
    df["intent"]
    .value_counts()
    .min()
)

print(
    "\nMinimum examples for one intent:",
    minimum_examples
)

if minimum_examples < 3:

    raise ValueError(
        "Each intent should contain at least "
        "3 examples."
    )

elif minimum_examples < 5:

    print(
        "\nWARNING:"
        "\nSome intents have fewer than 5 examples."
        "\nMore examples per intent will improve performance."
    )


# ============================================================
# 17. LABEL ENCODING
# ============================================================

label_encoder = LabelEncoder()

y = label_encoder.fit_transform(
    df["intent"]
)

number_of_classes = len(
    label_encoder.classes_
)

print(
    "\nNumber of classes:",
    number_of_classes
)

print("\nIntent encoding:")

for index, intent in enumerate(
    label_encoder.classes_
):

    print(
        f"{index} -> {intent}"
    )


# ============================================================
# 18. TOKENIZER
# ============================================================

MAX_WORDS = 5000

tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>",
    lower=True,
    filters=""
)

tokenizer.fit_on_texts(
    df["clean_text"]
)


# ============================================================
# 19. TEXT TO SEQUENCES
# ============================================================

sequences = tokenizer.texts_to_sequences(
    df["clean_text"]
)


# ============================================================
# 20. DETERMINE SEQUENCE LENGTH
# ============================================================

sequence_lengths = np.array([
    len(sequence)
    for sequence in sequences
])

percentile_95 = int(
    np.percentile(
        sequence_lengths,
        95
    )
)

MAX_LENGTH = max(
    12,
    min(
        30,
        percentile_95
    )
)

print(
    "\nSequence length:",
    MAX_LENGTH
)


# ============================================================
# 21. PAD SEQUENCES
# ============================================================

X = pad_sequences(

    sequences,

    maxlen=MAX_LENGTH,

    padding="post",

    truncating="post"
)

print(
    "Input shape:",
    X.shape
)


# ============================================================
# 22. TRAIN / VALIDATION / TEST SPLIT
# ============================================================

X_train, X_temp, y_train, y_temp = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=SEED,

    stratify=y
)


X_val, X_test, y_val, y_test = train_test_split(

    X_temp,

    y_temp,

    test_size=0.50,

    random_state=SEED,

    stratify=y_temp
)


print("\n" + "=" * 80)
print("TRAIN / VALIDATION / TEST SPLIT")
print("=" * 80)

print(
    "Training samples:",
    len(X_train)
)

print(
    "Validation samples:",
    len(X_val)
)

print(
    "Testing samples:",
    len(X_test)
)


# ============================================================
# 23. VOCABULARY SIZE
# ============================================================

vocab_size = min(
    MAX_WORDS,
    len(tokenizer.word_index) + 1
)

print(
    "\nVocabulary size:",
    vocab_size
)


# ============================================================
# 24. CLASS WEIGHTS
# ============================================================

classes = np.unique(y_train)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = {
    int(class_id): float(weight)
    for class_id, weight
    in zip(classes, class_weights_array)
}

print("\nClass weights:")
print(class_weights)


# ============================================================
# 25. BUILD BIDIRECTIONAL LSTM MODEL
# ============================================================

print("\n" + "=" * 80)
print("BUILDING BIDIRECTIONAL LSTM MODEL")
print("=" * 80)


model = Sequential([

    Input(
        shape=(MAX_LENGTH,)
    ),

    Embedding(

        input_dim=vocab_size,

        output_dim=64,

        mask_zero=True
    ),

    SpatialDropout1D(
        0.25
    ),

    Bidirectional(

        LSTM(
            64,
            return_sequences=False,
            dropout=0.20,
            recurrent_dropout=0.10
        )
    ),

    Dropout(
        0.40
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dropout(
        0.30
    ),

    Dense(
        number_of_classes,
        activation="softmax"
    )
])


# ============================================================
# 26. COMPILE
# ============================================================

model.compile(

    optimizer=Adam(
        learning_rate=0.001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=[
        "accuracy"
    ]
)


# ============================================================
# 27. MODEL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("MODEL SUMMARY")
print("=" * 80)

model.summary()


# ============================================================
# 28. CALLBACKS
# ============================================================

early_stopping = EarlyStopping(

    monitor="val_loss",

    patience=10,

    mode="min",

    restore_best_weights=True,

    verbose=1
)


reduce_lr = ReduceLROnPlateau(

    monitor="val_loss",

    factor=0.5,

    patience=3,

    min_lr=0.00001,

    verbose=1
)


checkpoint = ModelCheckpoint(

    "best_chatbot_lstm.keras",

    monitor="val_loss",

    save_best_only=True,

    mode="min",

    verbose=1
)


# ============================================================
# 29. TRAIN MODEL
# ============================================================

print("\n" + "=" * 80)
print("TRAINING BIDIRECTIONAL LSTM")
print("=" * 80)


history = model.fit(

    X_train,

    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=100,

    batch_size=8,

    class_weight=class_weights,

    callbacks=[
        early_stopping,
        reduce_lr,
        checkpoint
    ],

    verbose=1
)


# ============================================================
# 30. LOAD BEST MODEL
# ============================================================

best_model_path = (
    "best_chatbot_lstm.keras"
)

if os.path.exists(
    best_model_path
):

    model = tf.keras.models.load_model(
        best_model_path
    )


# ============================================================
# 31. EVALUATE MODEL
# ============================================================

print("\n" + "=" * 80)
print("LSTM MODEL EVALUATION")
print("=" * 80)


test_loss, test_accuracy = model.evaluate(

    X_test,

    y_test,

    verbose=0
)


print(
    "\nTest Loss:",
    round(
        float(test_loss),
        4
    )
)

print(
    "Test Accuracy:",
    round(
        float(test_accuracy) * 100,
        2
    ),
    "%"
)


# ============================================================
# 32. TEST PREDICTIONS
# ============================================================

lstm_predictions = model.predict(

    X_test,

    verbose=0
)


lstm_predicted_classes = np.argmax(

    lstm_predictions,

    axis=1
)


print("\nLSTM Classification Report:")

print(
    classification_report(

        y_test,

        lstm_predicted_classes,

        labels=np.arange(number_of_classes),

        target_names=label_encoder.classes_,

        zero_division=0
    )
)


# ============================================================
# 33. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(

    y_test,

    lstm_predicted_classes,

    labels=np.arange(number_of_classes)
)

print("\nConfusion Matrix:")
print(cm)


# ============================================================
# 34. CREATE RESPONSE DICTIONARY
# ============================================================

intent_responses = {}

for intent in label_encoder.classes_:

    responses = (
        df[
            df["intent"] == intent
        ]["response"]
        .tolist()
    )

    intent_responses[
        intent
    ] = responses


# ============================================================
# 35. BUILD TF-IDF MODEL
# ============================================================

print("\n" + "=" * 80)
print("BUILDING TF-IDF SIMILARITY MODEL")
print("=" * 80)


tfidf_vectorizer = TfidfVectorizer(

    lowercase=True,

    ngram_range=(1, 2),

    sublinear_tf=True,

    min_df=1
)


tfidf_matrix = tfidf_vectorizer.fit_transform(

    df["clean_text"]
)


# ============================================================
# 36. TF-IDF PREDICTION
# ============================================================

def tfidf_similarity_prediction(
    user_input
):

    clean_input = preprocess_text(
        user_input
    )

    if not clean_input:

        return (
            None,
            0.0,
            None
        )


    user_vector = (
        tfidf_vectorizer
        .transform(
            [clean_input]
        )
    )


    similarity_scores = (
        tfidf_matrix
        .dot(
            user_vector.T
        )
        .toarray()
        .flatten()
    )


    best_index = int(
        np.argmax(
            similarity_scores
        )
    )


    best_score = float(
        similarity_scores[
            best_index
        ]
    )


    best_intent = (
        df.iloc[
            best_index
        ]["intent"]
    )


    return (
        best_intent,
        best_score,
        best_index
    )


# ============================================================
# 37. LSTM PREDICTION
# ============================================================

def lstm_prediction(
    user_input
):

    clean_input = preprocess_text(
        user_input
    )

    if not clean_input:

        return (
            None,
            0.0,
            []
        )


    sequence = (
        tokenizer
        .texts_to_sequences(
            [clean_input]
        )
    )


    padded_sequence = pad_sequences(

        sequence,

        maxlen=MAX_LENGTH,

        padding="post",

        truncating="post"
    )


    prediction = model.predict(

        padded_sequence,

        verbose=0
    )[0]


    predicted_class = int(
        np.argmax(
            prediction
        )
    )


    confidence = float(
        prediction[
            predicted_class
        ]
    )


    predicted_intent = (
        label_encoder
        .inverse_transform(
            [predicted_class]
        )[0]
    )


    top_indices = np.argsort(
        prediction
    )[-3:][::-1]


    top_predictions = []


    for index in top_indices:

        intent = (
            label_encoder
            .inverse_transform(
                [int(index)]
            )[0]
        )


        score = float(
            prediction[
                index
            ]
        )


        top_predictions.append(
            (
                intent,
                score
            )
        )


    return (
        predicted_intent,
        confidence,
        top_predictions
    )


# ============================================================
# 38. HYBRID INTENT PREDICTION
# ============================================================

def predict_intent(
    user_input
):

    clean_input = preprocess_text(
        user_input
    )

    if not clean_input:

        return (
            None,
            0.0,
            [],
            0.0,
            "Invalid input"
        )


    # --------------------------------------------------------
    # LSTM
    # --------------------------------------------------------

    (
        lstm_intent,
        lstm_confidence,
        top_predictions
    ) = lstm_prediction(
        user_input
    )


    # --------------------------------------------------------
    # TF-IDF
    # --------------------------------------------------------

    (
        tfidf_intent,
        tfidf_score,
        tfidf_index
    ) = tfidf_similarity_prediction(
        user_input
    )


    # --------------------------------------------------------
    # HYBRID LOGIC
    # --------------------------------------------------------
    #
    # Priority:
    #
    # 1. Exact text match
    # 2. Strong LSTM + supporting TF-IDF
    # 3. Strong TF-IDF similarity
    # 4. LSTM
    # 5. Unknown
    #
    # TF-IDF should NOT automatically override
    # the LSTM.
    # --------------------------------------------------------


    exact_match_indices = np.where(

        df["clean_text"].values
        == clean_input

    )[0]


    # --------------------------------------------------------
    # EXACT MATCH
    # --------------------------------------------------------

    if len(
        exact_match_indices
    ) > 0:

        exact_index = int(
            exact_match_indices[0]
        )

        final_intent = (
            df.iloc[
                exact_index
            ]["intent"]
        )

        decision = (
            "Exact training-example match"
        )

        return (

            final_intent,

            lstm_confidence,

            top_predictions,

            1.0,

            decision
        )


    # --------------------------------------------------------
    # STRONG LSTM
    # --------------------------------------------------------

    if lstm_confidence >= 0.75:

        final_intent = lstm_intent

        decision = (
            "Strong LSTM prediction"
        )

        return (

            final_intent,

            lstm_confidence,

            top_predictions,

            tfidf_score,

            decision
        )


    # --------------------------------------------------------
    # LSTM + TF-IDF AGREEMENT
    # --------------------------------------------------------

    if (
        lstm_intent == tfidf_intent
        and
        lstm_confidence >= 0.35
        and
        tfidf_score >= 0.20
    ):

        final_intent = lstm_intent

        decision = (
            "LSTM + TF-IDF agreement"
        )

        return (

            final_intent,

            lstm_confidence,

            top_predictions,

            tfidf_score,

            decision
        )


    # --------------------------------------------------------
    # STRONG TF-IDF
    # --------------------------------------------------------

    if tfidf_score >= 0.65:

        final_intent = tfidf_intent

        decision = (
            "Strong TF-IDF similarity"
        )

        return (

            final_intent,

            lstm_confidence,

            top_predictions,

            tfidf_score,

            decision
        )


    # --------------------------------------------------------
    # MODERATE LSTM
    # --------------------------------------------------------

    if lstm_confidence >= 0.45:

        final_intent = lstm_intent

        decision = (
            "Moderate LSTM prediction"
        )

        return (

            final_intent,

            lstm_confidence,

            top_predictions,

            tfidf_score,

            decision
        )


    # --------------------------------------------------------
    # MODERATE TF-IDF
    # --------------------------------------------------------

    if tfidf_score >= 0.30:

        final_intent = tfidf_intent

        decision = (
            "Moderate TF-IDF similarity"
        )

        return (

            final_intent,

            lstm_confidence,

            top_predictions,

            tfidf_score,

            decision
        )


    # --------------------------------------------------------
    # UNKNOWN
    # --------------------------------------------------------

    decision = (
        "Low confidence - unknown intent"
    )

    return (

        None,

        lstm_confidence,

        top_predictions,

        tfidf_score,

        decision
    )


# ============================================================
# 39. CHATBOT RESPONSE
# ============================================================

def chatbot_response(
    user_input
):

    if not user_input.strip():

        return (
            "Please enter a question."
        )


    (
        intent,
        lstm_confidence,
        top_predictions,
        tfidf_score,
        decision
    ) = predict_intent(
        user_input
    )


    print(
        f"\n[Final intent: {intent}]"
    )

    print(
        f"[LSTM confidence: "
        f"{lstm_confidence:.2%}]"
    )

    print(
        f"[TF-IDF similarity: "
        f"{tfidf_score:.2%}]"
    )

    print(
        f"[Decision: {decision}]"
    )


    print(
        "\nTop LSTM predictions:"
    )


    for predicted_intent, score in (
        top_predictions
    ):

        print(
            f"  {predicted_intent}: "
            f"{score:.2%}"
        )


    # --------------------------------------------------------
    # UNKNOWN INTENT
    # --------------------------------------------------------

    if intent is None:

        return (
            "I'm not completely sure what you mean. "
            "I can help with Python, AI, machine learning, "
            "deep learning, NLP, RNN, LSTM, CNN, "
            "Transformers, courses, enrollment, fees, "
            "certificates and related topics."
        )


    # --------------------------------------------------------
    # GET RESPONSES
    # --------------------------------------------------------

    responses = intent_responses.get(

        intent,

        []
    )


    if not responses:

        return (
            "I don't have a response for "
            "that question yet."
        )


    # --------------------------------------------------------
    # DETERMINISTIC RESPONSE
    # --------------------------------------------------------

    # Use a deterministic response instead of
    # random.choice() so that the chatbot behaves
    # consistently.

    clean_input = preprocess_text(
        user_input
    )

    response_index = (
        sum(
            ord(character)
            for character in clean_input
        )
        % len(responses)
    )


    return responses[
        response_index
    ]


# ============================================================
# 40. TEST QUESTIONS
# ============================================================

print("\n" + "=" * 80)
print("TESTING CHATBOT")
print("=" * 80)


test_questions = [

    "Hello",

    "Hi",

    "Hey",

    "Good morning",

    "What is Python?",

    "Tell me about Python",

    "Explain Python",

    "What is machine learning?",

    "Tell me about machine learning",

    "Explain machine learning",

    "What is deep learning?",

    "What is RNN?",

    "Explain RNN",

    "What is LSTM?",

    "Explain LSTM",

    "What is CNN?",

    "What is NLP?",

    "Explain NLP",

    "What is a transformer?",

    "What is artificial intelligence?",

    "Thank you",

    "Thanks",

    "Bye",

    "Goodbye",

    "What is Java?",

    "Tell me a joke",

    "Who is Sachin Tendulkar?"

]


for question in test_questions:

    print(
        "\n" + "-" * 80
    )

    print(
        "User:",
        question
    )


    response = chatbot_response(
        question
    )


    print(
        "Bot:",
        response
    )


# ============================================================
# 41. SAVE LSTM MODEL
# ============================================================

model.save(
    "chatbot_lstm.keras"
)

print(
    "\nLSTM model saved:"
)

print(
    "chatbot_lstm.keras"
)


# ============================================================
# 42. SAVE TOKENIZER
# ============================================================

with open(
    "chatbot_tokenizer.pkl",
    "wb"
) as file:

    pickle.dump(
        tokenizer,
        file
    )

print(
    "Tokenizer saved:"
)

print(
    "chatbot_tokenizer.pkl"
)


# ============================================================
# 43. SAVE LABEL ENCODER
# ============================================================

with open(
    "chatbot_label_encoder.pkl",
    "wb"
) as file:

    pickle.dump(
        label_encoder,
        file
    )

print(
    "Label encoder saved:"
)

print(
    "chatbot_label_encoder.pkl"
)


# ============================================================
# 44. SAVE TF-IDF VECTORIZER
# ============================================================

with open(
    "chatbot_tfidf_vectorizer.pkl",
    "wb"
) as file:

    pickle.dump(
        tfidf_vectorizer,
        file
    )

print(
    "TF-IDF vectorizer saved:"
)

print(
    "chatbot_tfidf_vectorizer.pkl"
)


# ============================================================
# 45. SAVE PROCESSED DATASET
# ============================================================

df.to_csv(

    "chatbot_processed_dataset.csv",

    index=False
)

print(
    "Processed dataset saved:"
)

print(
    "chatbot_processed_dataset.csv"
)


# ============================================================
# 46. SAVE MODEL INFORMATION
# ============================================================

model_info = {

    "tensorflow_version": tf.__version__,

    "number_of_samples":
        int(len(df)),

    "number_of_intents":
        int(number_of_intents),

    "number_of_classes":
        int(number_of_classes),

    "vocabulary_size":
        int(vocab_size),

    "max_length":
        int(MAX_LENGTH),

    "test_accuracy":
        float(test_accuracy),

    "seed":
        SEED
}


with open(
    "chatbot_model_info.pkl",
    "wb"
) as file:

    pickle.dump(
        model_info,
        file
    )


# ============================================================
# 47. SAVED FILES
# ============================================================

print("\n" + "=" * 80)
print("SAVED FILES")
print("=" * 80)

print("\n1. chatbot_lstm.keras")

print("2. best_chatbot_lstm.keras")

print("3. chatbot_tokenizer.pkl")

print("4. chatbot_label_encoder.pkl")

print("5. chatbot_tfidf_vectorizer.pkl")

print("6. chatbot_processed_dataset.csv")

print("7. chatbot_model_info.pkl")


# ============================================================
# 48. START INTERACTIVE CHATBOT
# ============================================================

print("\n" + "=" * 80)
print("🤖 AI CHATBOT ASSISTANT IS READY")
print("=" * 80)

print("\nCommands:")

print(
    "  exit     -> Stop chatbot"
)

print(
    "  intents  -> Show available intents"
)

print(
    "  test     -> Run sample questions"
)

print("=" * 80)


while True:

    user_input = input(
        "\nYou: "
    ).strip()


    # --------------------------------------------------------
    # EXIT
    # --------------------------------------------------------

    if user_input.lower() in [
        "exit",
        "quit"
    ]:

        print(
            "Bot: Goodbye! Have a nice day!"
        )

        break


    # --------------------------------------------------------
    # SHOW INTENTS
    # --------------------------------------------------------

    if user_input.lower() == "intents":

        print(
            "\nAvailable intents:"
        )

        for intent in (
            label_encoder.classes_
        ):

            print(
                "-",
                intent
            )

        continue


    # --------------------------------------------------------
    # TEST
    # --------------------------------------------------------

    if user_input.lower() == "test":

        print(
            "\nRunning sample questions..."
        )

        for question in test_questions:

            print(
                "\nUser:",
                question
            )

            response = chatbot_response(
                question
            )

            print(
                "Bot:",
                response
            )

        continue


    # --------------------------------------------------------
    # EMPTY INPUT
    # --------------------------------------------------------

    if user_input == "":

        print(
            "Bot: Please enter a message."
        )

        continue


    # --------------------------------------------------------
    # NORMAL CHAT
    # --------------------------------------------------------

    response = chatbot_response(
        user_input
    )

    print(
        "Bot:",
        response
    )

                     AI CHATBOT ASSISTANT
          TENSORFLOW + NLP + BIDIRECTIONAL LSTM

TensorFlow version: 2.20.0

Upload your CSV file.

Required columns:
intent, text, response



Saving ai_chatbot_dataset.csv to ai_chatbot_dataset (5).csv

Uploaded file: ai_chatbot_dataset (5).csv

CSV loaded successfully!
Original dataset shape: (203, 3)

Original columns:
['intent', 'text', 'response']

Missing values:
intent      0
text        0
response    0
dtype: int64

DATASET INFORMATION

Dataset shape: (203, 4)

Number of intents: 20

Intent distribution:
intent
greeting            15
goodbye             10
help                10
about_bot           10
nlp                 10
python              10
machine_learning    10
deep_learning       10
neural_network      10
rnn                 10
lstm                10
cnn                 10
enrollment          10
transformer         10
ai                  10
course              10
fees                10
certificate         10
contact             10
thanks               8
Name: count, dtype: int64

Minimum examples for one intent: 8

Number of classes: 20

Intent encoding:
0 -> about_bot
1 -> ai
2 -> certificate
3 -> cnn
4 -> c

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 12, 64)         │        13,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ (None, 12, 64)         │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 20)             │         1,300 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 89,172 (348.33 KB)

 Trainable params: 89,172 (348.33 KB)

 Non-trainable params: 0 (0.00 B)


TRAINING BIDIRECTIONAL LSTM
Epoch 1/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.0440 - loss: 2.9974
Epoch 1: val_loss improved from None to 2.99173, saving model to best_chatbot_lstm.keras

Epoch 1: finished saving model to best_chatbot_lstm.keras
21/21 ━━━━━━━━━━━━━━━━━━━━ 16s 91ms/step - accuracy: 0.0556 - loss: 2.9969 - val_accuracy: 0.0500 - val_loss: 2.9917 - learning_rate: 0.0010
Epoch 2/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.1153 - loss: 2.9863
Epoch 2: val_loss improved from 2.99173 to 2.98552, saving model to best_chatbot_lstm.keras

Epoch 2: finished saving model to best_chatbot_lstm.keras
21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.1049 - loss: 2.9889 - val_accuracy: 0.2000 - val_loss: 2.9855 - learning_rate: 0.0010
Epoch 3/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.1468 - loss: 2.9799
Epoch 3: val_loss improved from 2.98552 to 2.97507, saving model to best_chatbot_lstm.keras

Epoch 3: finished saving model to best_c